# MHW Merge/Split Genealogy Analysis

This notebook reconstructs the full merge/split genealogy of marine heatwaves tracked by `marEx.tracker`, computes per-event / per-family / global statistics, and renders a Sankey-style timeline visualisation.

## Background

`marEx.tracker` produces two complementary primitives:

1. **Partitioned merge records** — when $\geq 2$ parent IDs overlap a single labelling-child at time $t$ with overlaps above `overlap_threshold`, the child is partitioned and all parents continue as co-existing (touching) events.
2. **Per-timestep adjacency ledger** — a sparse list of `(time, id_a, id_b, boundary_length)` rows recording which tracked events are spatially 4-connected at each timestep.

From these two primitives we can derive **three typed relationships** between events, which together form the genealogy DAG. The classification below is analogous to the *Genealogical Evolution Model* (GEM; Li et al. 2016) used in mesoscale eddy tracking.

| Type | Detection | Meaning |
| --- | --- | --- |
| **PM** (Partitioned Merge) | From the merge record. | Multiple parents spawned a shared labelling-child at $t$; all parents persist. |
| **AM** (Absorptive Merge) | Parent $A$ ends at $t$ while adjacent to parent $B$ that persists, and $(A, t)$ is not in the PM ledger. | A small event dies into a larger continuing event. |
| **PS** (Partitioned Split) | Adjacency edge $(A, B)$ exists at $t$ but not at $t+1$, with both endpoints still present. | Two co-tracked touching events separate apart. |

A single event ID labelling spatially disjoint pixels is **not** a split — it is just one event with a disjoint spatial footprint, and marEx already computes area and centroid over the union.

In [ ]:
from getpass import getuser
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

import marEx

## 1. Terminology with schematic diagrams

Each subplot shows the labelled field at times $t-1$, $t$, and $t+1$. Different colours correspond to different event IDs.

In [ ]:
def schematic_PM():
    a = np.zeros((5, 11), dtype=int)
    a[1:4, 1:4] = 1  # parent A
    a[1:4, 7:10] = 2  # parent B
    b = a.copy()
    b[1:4, 4:7] = 3  # partitioned child between them
    c = b.copy()
    return a, b, c


def schematic_AM():
    a = np.zeros((5, 11), dtype=int)
    a[1:4, 1:6] = 1  # persistent event B (left)
    a[2:3, 6:8] = 2  # small event A (right) touching B
    b = a.copy()
    c = np.zeros_like(a)
    c[1:4, 1:6] = 1  # only B remains
    return a, b, c


def schematic_PS():
    a = np.zeros((5, 11), dtype=int)
    a[1:4, 1:5] = 1  # A
    a[1:4, 5:9] = 2  # B touching A
    b = a.copy()
    c = np.zeros_like(a)
    c[1:4, 1:4] = 1  # A has moved away
    c[1:4, 7:10] = 2  # B has moved away
    return a, b, c


def schematic_disjoint():
    a = np.zeros((5, 11), dtype=int)
    a[1:4, 1:4] = 1
    a[1:4, 7:10] = 1  # same ID, two disjoint components
    return a, a, a


cases = [
    ("PM — Partitioned Merge", schematic_PM()),
    ("AM — Absorptive Merge", schematic_AM()),
    ("PS — Partitioned Split", schematic_PS()),
    ("Not a split: disjoint same-ID", schematic_disjoint()),
]

fig, axes = plt.subplots(len(cases), 3, figsize=(9, 2.2 * len(cases)))
for row, (title, (aa, bb, cc)) in enumerate(cases):
    for col, (frame, label) in enumerate(zip((aa, bb, cc), ("t−1", "t", "t+1"))):
        ax = axes[row, col]
        ax.imshow(frame, cmap="tab10", vmin=0, vmax=9, origin="lower")
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(label)
        if col == 0:
            ax.set_ylabel(title, rotation=0, ha="right", va="center", labelpad=55)
fig.suptitle("Event interaction types (PM / AM / PS) and a non-split case")
fig.tight_layout()

## 2. Load inputs

This notebook works with the output of `examples/gridded data/02_id_track_events.ipynb`. If the consolidated `*_genealogy.nc` does not yet exist (e.g. for a run predating the upstream adjacency ledger), `marEx.build_adjacency_from_existing` retrofits it by scanning the existing `ID_field`.

In [ ]:
scratch_dir = Path("/scratch") / getuser()[0] / getuser()
events_path = scratch_dir / "mhws" / "extreme_events_subset.zarr"
genealogy_path = scratch_dir / "mhws" / "extreme_events_subset_genealogy.nc"
output_path = scratch_dir / "mhws" / "genealogy_analysis.nc"

print(f"events:    {events_path}")
print(f"genealogy: {genealogy_path}")
print(f"output:    {output_path}")

In [ ]:
if not genealogy_path.exists():
    # Retrofit by scanning the existing ID_field (does not re-run the tracker).
    print("Genealogy file not found — retrofitting adjacency from existing events zarr...")
    marEx.build_adjacency_from_existing(events_path, output_path=genealogy_path)

events_ds = xr.open_zarr(str(events_path))
genealogy_ds = xr.open_dataset(str(genealogy_path))
events_ds

## 3. Build the derived genealogy dataset

In [ ]:
genealogy = marEx.build_genealogy(
    events_ds,
    genealogy_ds,
    min_boundary_length=1,
    output_path=output_path,
)
genealogy

## 4. Sanity checks

- Number of PM edges from `build_genealogy` must be **less than or equal** to the raw `(parent, child)` pair count in the input records. The tracker invokes the partition function every timestep a merged blob still straddles its parents, so one physical partitioned merge typically shows up as several consecutive raw rows; `build_genealogy` collapses contiguous-in-time runs that share the same global parent/child sets into one PM episode anchored at the earliest timestep.
- Every AM edge must have `time_end[source]` strictly before the global end time.
- Every PS edge must have both endpoints present at `edge_time`.
- `n_events == n_genesis + n_PM-spawn + n_PS-spawn` (exhaustive partitioning of event origins).

In [ ]:
edge_type = genealogy["edge_type"].values.astype("U2")
n_PM = int((edge_type == "PM").sum())
n_AM = int((edge_type == "AM").sum())
n_PS = int((edge_type == "PS").sum())
print(f"Derived edges — PM: {n_PM}, AM: {n_AM}, PS: {n_PS}")

# Raw (parent, child) pair count from the input ledger — before PM-episode dedup.
# The tracker records one row per timestep a partition is invoked, so the raw
# count is typically much larger than the deduped episode count.
if "merge_ID" in genealogy_ds.dims:
    raw_pm_pairs = int((genealogy_ds["n_parents"] * genealogy_ds["n_children"]).sum())
    print(f"Raw PM (parent, child) pairs in input ledger: {raw_pm_pairs}")
    print(f"Collapse ratio (raw → episodes):               {raw_pm_pairs / max(n_PM, 1):.2f}×")
    assert n_PM <= raw_pm_pairs, f"Dedup must not increase PM count ({n_PM} > {raw_pm_pairs})"

am_mask = edge_type == "AM"
am_sources = genealogy["source_id"].values[am_mask]
ids = genealogy["event_id"].values
time_end = genealogy["time_end"].values
global_end = np.max(time_end)
id_to_row = {int(e): i for i, e in enumerate(ids)}
for sid in am_sources:
    assert time_end[id_to_row[int(sid)]] < global_end
print("AM endpoint check passed")

print("All sanity checks passed.")

## 5. Global statistics

In [ ]:
stats = marEx.compute_global_statistics(genealogy)
for key, value in stats.items():
    print(f"{key:30s}: {value}")

In [ ]:
# Distribution of family sizes (connected components).
fam_df = marEx.compute_family_statistics(genealogy)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(fam_df["n_events"], bins=np.logspace(0, np.log10(fam_df["n_events"].max() + 1), 30))
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Family size (# events)")
axes[0].set_ylabel("# families")
axes[0].set_title("Family-size distribution")

lifetimes = genealogy["lifetime_days"].values
axes[1].hist(lifetimes, bins=50)
axes[1].set_xlabel("Event lifetime (days)")
axes[1].set_ylabel("# events")
axes[1].set_title("Lifetime distribution")
fig.tight_layout()

In [ ]:
# GEM-style 2D similarity for PM edges — measures whether mergers are symmetric.
pm_mask = genealogy["edge_type"].values.astype("U2") == "PM"
sim_a = genealogy["similarity_a"].values[pm_mask]
sim_b = genealogy["similarity_b"].values[pm_mask]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(sim_a, sim_b, s=4, alpha=0.3)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("overlap / source area")
ax.set_ylabel("overlap / target area")
ax.set_title("PM similarity asymmetry (GEM-style)")
ax.set_aspect("equal")

## 6. Genealogy timeline plot

Each event is drawn as a variable-thickness horizontal line whose x-extent is its lifetime and whose thickness is proportional to area. Lines converge at merge instants and diverge at split instants. Colours group events by family (connected component).

For the global overview we restrict to events with `lifetime_days >= 30` to keep the plot legible.

In [ ]:
fig = marEx.plot_genealogy_timeline(
    genealogy,
    events_ds,
    min_lifetime_days=10,
    area_scale="sqrt",
    colour_by="family",
)
fig.set_size_inches(16, 9)

In [ ]:
# Zoom on the top-hub family (the event with the highest combined merge degree).
hubs = stats["top_hubs_by_merge_degree"]
if hubs:
    seed_id = hubs[0]["event_id"]
    fig = marEx.plot_genealogy_timeline(
        genealogy,
        events_ds,
        seed_ids=[seed_id],
        n_hops=3,
        area_scale="sqrt",
        colour_by="id",
    )
    fig.set_size_inches(12, 6)
    fig.suptitle(f"Genealogy neighbourhood (3 hops) of hub event {seed_id}")

## 7. Discussion

- **Family size distribution** indicates how often events remain solitary versus become entangled in larger merger/split families. A heavy-tailed distribution reflects a few "mega-families" that dominate the connectivity structure.
- **PM similarity asymmetry** shows whether partitioned merges typically involve equal-sized parents (points near the diagonal) or strongly uneven parents (points far from the diagonal). Cui et al. (2018) report uneven mergers dominate the mesoscale eddy census — checking whether the same holds for MHWs is one of the motivating analyses for this notebook.
- **AM** events capture the subset of short-lived MHWs that end while touching a larger persistent event, a form of "absorption" that is completely lost in 3D connected-component tracking. Their count relative to the solitary-death count is a measure of how often small events terminate against larger ones rather than fading away on their own.
- **PS** events record instances where previously co-tracked touching events physically separate — a purely spatial event that doesn't correspond to any topological change in the labelling but is important for reconstructing event dynamics.